In [ ]:
#IMPORTING LIBRARIES
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#LOADING THE DATASET
file_path = r"../data/Bosch_Demand_Datasheet.xlsx"

df = pd.read_excel(file_path, sheet_name="Basic Plus Red")

df.head(20)

In [ ]:
df = df.rename(
    columns = {
        "Year" : "Date",
        "Demand: Red" : "Demand"
    }
)

df["Date"] = pd.to_datetime(df["Date"])
df.head(10)

In [ ]:
df["Demand"].describe()

In [ ]:
#PLOTTING ACTUAL DEMAND
plt.figure(figsize=(12,5))
plt.plot(df["Date"], df["Demand"])
plt.title("Basic Plus Red Demand")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.grid(True)
plt.show()

In [ ]:
first_12_months = df["Demand"].iloc[:12]
initial_level = first_12_months.mean()
print(initial_level)

In [ ]:
#MANUALLY APPLYING SIMPLE EXPONENTIAL SMOOTHING
df["F-SES"] = None
df.loc[12,"F-SES"] = initial_level

alpha = 0.8

for i in range(13,len(df)):
    df.loc[i,"F-SES"] = df.loc[i-1,"F-SES"] + alpha*(df.loc[i-1,"Demand"] - df.loc[i-1,"F-SES"])

df.head(20)

In [ ]:
#CALCULATION OF MEAN ABSOLUTE ERROR AND MEAN ABSOLUTE PERCENTAGE ERROR
df["Error"] = df["Demand"] - df["F-SES"]
df["Abs_Error"] = abs(df["Error"])

mae = df["Abs_Error"].mean()
print(mae)

df.iloc[0:21]

df["Abs_Pct_Error"] = (df["Abs_Error"]/df["Demand"])*100

mape = df["Abs_Pct_Error"].mean()
print(mape)

#print(df["Abs_Error"].count())
#print(len(df))


In [ ]:
#MANUALLY CALCULATING ALPHA
df_alpha01 = df[["Date", "Demand"]].copy()
df_alpha01["Forecast"] = None
df_alpha01.loc[12,"Forecast"] = 104.75
df_alpha01.head(20)

alpha = 0.1
for i in range(13,len(df_alpha01)):
    df_alpha01.loc[i,"Forecast"] = df_alpha01.loc[i-1,"Forecast"] + alpha * (df_alpha01.loc[i-1,"Demand"] - df_alpha01.loc[i-1,"Forecast"])

df_alpha01.iloc[10:21]


In [ ]:
#ERROR CALCULATIONS FOR ALPHA = 0.1
df_alpha01["Error"] = df_alpha01["Demand"] - df_alpha01["Forecast"]
df_alpha01["Abs_Error"] = abs(df_alpha01["Error"])
df_alpha01["Abs_Pct_Error"] = (df_alpha01["Abs_Error"]/df_alpha01["Demand"])*100

mae_alpha01 = df_alpha01["Abs_Error"].mean()
mape_alpha01 = df_alpha01["Abs_Pct_Error"].mean()

print(mae_alpha01)
print(mape_alpha01)

df_alpha01.iloc[0:21]

In [ ]:
print("MAE difference =", mae - mae_alpha01)
print("MAPE difference =", mape - mape_alpha01)

In [ ]:
#MANUAL SELECTION OF AN OPTIMAL ALPHA
results = []
alphas = [0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,0.10]

for alpha in alphas:
    temp_df = df[["Date", "Demand"]].copy()
    temp_df["Forecast"] = None
    temp_df.loc[12,"Forecast"] = 104.75

    for i in range(13,len(temp_df)):
        temp_df.loc[i,"Forecast"] = temp_df.loc[i-1,"Forecast"] + alpha * (temp_df.loc[i-1,"Demand"] - temp_df.loc[i-1,"Forecast"])

        temp_df["Error"] = temp_df["Demand"] - temp_df["Forecast"]
        temp_df["Abs_Error"] = abs(temp_df["Error"])
        temp_df["Abs_Pct_Error"] = (temp_df["Abs_Error"]/temp_df["Demand"])*100

    mae = temp_df["Abs_Error"].mean()
    mape = temp_df["Abs_Pct_Error"].mean()
    results.append([alpha, mae, mape])


results_df = pd.DataFrame(results,columns=["Alpha", "MAE", "MAPE"])
results_df.head(30)

In [ ]:
#USING STATSMODELS TO FIT THE MODEL AND SELECTING MOST OPTIMUM ALPHA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
model = SimpleExpSmoothing(df["Demand"]).fit(
    optimized=True
)

df["SES_fitted"] = model.fittedvalues
plt.figure(figsize = (12,5))
plt.plot(df["Date"], df["Demand"], label="Actual Demand")
plt.plot(df["Date"], df["SES_fitted"], label="SES Fitted Demand")
plt.title("Basic Plus Red - Actual vs SES Fitted Demand")
plt.legend()
plt.grid(True)
plt.show()

df["Error"] = df["Demand"] - df["SES_fitted"]
df["Abs_Error"] = abs(df["Error"])

mae = df["Abs_Error"].mean()
print("MAE for SES: ", mae)

df["Abs_Pct_Error"] = (df["Abs_Error"]/df["Demand"])*100
mape = df["Abs_Pct_Error"].mean()
print("MAPE for SES: ", mape,"%")


In [ ]:
#EXPLORATORY ANALYSIS
print("Alpha =", model.params["smoothing_level"])
print(model.params)

In [ ]:
#EXPLORATORY ANALYSIS
print(model.fittedvalues.head(10))
print(model.forecast(5))
dir(model)
type(model.params)
type(model.fittedvalues)
model.fittedvalues.describe()